# LT Signals Predictivity (Section 5.1)

Computes ROC-AUC of all metrics against correctness across models and datasets.
Saves results to `results/auc_all.csv`.

**Metrics compared:**
- LT signals (Section 3.1): `net_change`, `cumulative_change`, `aligned_change`
- Cross-layer baselines (Wang et al.): `layerwise_mag`, `layerwise_ang`
- Output-distribution baselines: `logit_margin`, `entropy`, `perplexity`

Note: output-distribution baselines require `compute_early_exit_properties.py` to have
been run. For models/datasets without an early-exit file (BigBench, deepseek-r1-llama70B/llama8B/qwen32B),
only the first 5 metrics are available.

In [6]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
import torch

import sys
sys.path.insert(0, '../src')
from post_utils import create_internals_df, load_internal_metrics

In [ ]:
PATH = Path('../')  # project root — adjust if running from a different location

## Configuration

In [8]:
MODELS = [
    'phi4-reasoning-plus',
    'deepseek-r1-qwen14B',
    'qwen3-14B',
    'deepseek-r1-llama8B',
    'deepseek-r1-qwen32B',
    'deepseek-r1-llama70B',
]

DATASETS = ['GPQA', 'AIME2025', 'TSP', 'bigbench/fables', 'bigbench/social-iqa']

# Segment length used to compute internal representations.
# Selects which *_ntokens_{N}_internals.pt file to load.
# Available values: 100, 300, 500, 700
# Note: bigbench and TSP for deepseek-r1-llama8B/qwen32B only have 100 and 300;
#       those combinations fall back to ntokens_300 automatically regardless of this setting.
AVERAGE_TYPE = 'ntokens_500'

# All 8 paper metrics
ALL_METRICS = [
    'net_change',        # LT signal
    'cumulative_change', # LT signal
    'aligned_change',    # LT signal
    'layerwise_mag',     # Wang et al. baseline
    'layerwise_ang',     # Wang et al. baseline
    'logit_margin',      # output-distribution baseline (requires early exit)
    'entropy',           # output-distribution baseline (requires early exit)
    'perplexity',        # output-distribution baseline (requires early exit)
]

# Metrics where higher value predicts lower accuracy — negate before AUC
NEGATE = {'cumulative_change', 'layerwise_ang'}

# Human-readable labels for display
LABEL_MAP = {
    'net_change':        'Net Change',
    'cumulative_change': 'Cumulative Change',
    'aligned_change':    'Aligned Change',
    'layerwise_mag':     'Layer Mag',
    'layerwise_ang':     'Layer Ang',
    'logit_margin':      'Logit Margin',
    'entropy':           'Entropy',
    'perplexity':        'Perplexity',
}

## Data loading

Two loading paths:
- **With early exit** (`create_internals_df`): used when `compute_early_exit_properties.py` was run for this model/dataset. Provides all 8 metrics. Used for phi4-reasoning-plus, deepseek-r1-qwen14B, qwen3-14B on GPQA/AIME2025/TSP.
- **Without early exit** (direct `.pt` load): used for BigBench (all models), deepseek-r1-llama70B/llama8B/qwen32B (all datasets). Accuracy is read from the internals file directly. Provides 5 metrics only.

Note: bigbench and TSP for deepseek-r1-llama8B/qwen32B only have `ntokens_300` results; all other combinations use `ntokens_500`.

In [9]:
def load_with_early_exit(model_name, dataset):
    """Load internals + early-exit data. Returns a single concatenated DataFrame."""
    internals_df, early_exit_df = create_internals_df(
        model_name=model_name,
        dataset=dataset,
        path=PATH,
        average_type=AVERAGE_TYPE,
    )
    internals_df = internals_df.groupby(
        ['datapoint', 'data_repeat', 'metric_name']
    ).mean(numeric_only=True).reset_index()
    internals_df['accuracy'] = internals_df['accuracy'] > 0
    return pd.concat([internals_df, early_exit_df], ignore_index=True), ALL_METRICS


def _load_acc_map(model_name, dataset):
    """Load accuracy from an _only_final early-exit file.

    Returns {(data_point_id, data_repeat_id): acc_at_100pct}.
    Used when the internals .pt file was produced without storing accuracy.
    """
    f = PATH / f"results/early_exit/{dataset}/{model_name}_output_properties_only_final.json"
    with open(f) as fh:
        df = pd.DataFrame(json.load(fh))
    return dict(zip(zip(df['data_point_id'], df['data_repeat_id']), df['acc_100']))


def load_without_early_exit(model_name, dataset, average_type=None):
    """Load internals only; accuracy is read from the .pt file or a fallback acc file."""
    if average_type is None:
        average_type = AVERAGE_TYPE
    lt_wang_metrics = [
        'net_change', 'cumulative_change', 'aligned_change',
        'layerwise_mag', 'layerwise_ang',
    ]
    trajectory_metrics = {'layerwise_mag', 'layerwise_ang', 'seqwise_mag', 'seqwise_ang'}

    internals = load_internal_metrics(model_name, dataset, PATH, average_type=average_type)

    # Some internals files don't store accuracy; fall back to the _only_final early-exit file.
    acc_map = {}
    if internals and 'accuracy' not in internals[0]:
        acc_map = _load_acc_map(model_name, dataset)

    rows = []
    for i in internals:
        acc = i['accuracy'] if 'accuracy' in i else acc_map.get(
            (i['data_point_id'], i['data_repeat_id']), 0
        )
        base = {
            'datapoint':      i['data_point_id'],
            'data_repeat':    i['data_repeat_id'],
            'accuracy':       acc > 0,
            'n_think_tokens': i['n_think_tokens'],
        }
        for metric in lt_wang_metrics:
            if metric in trajectory_metrics:
                rows.append({**base, 'metric_name': metric, 'layer': 'all',
                             'value': torch.mean(i[metric]).item()})
            else:
                for layer_idx, val in enumerate(i[metric]):
                    rows.append({**base, 'metric_name': metric, 'layer': layer_idx, 'value': val})

    df = pd.DataFrame(rows)
    df = df.groupby(['datapoint', 'data_repeat', 'metric_name']).mean(numeric_only=True).reset_index()
    df['accuracy'] = df['accuracy'] > 0
    return df, lt_wang_metrics


# Models that only have ntokens_300 results (no ntokens_500)
_NTOKENS_300_MODELS = {'deepseek-r1-llama8B', 'deepseek-r1-qwen32B'}

# Models without a full per-percentage early-exit file
_NO_EARLY_EXIT_MODELS = {'deepseek-r1-llama70B', 'deepseek-r1-llama8B', 'deepseek-r1-qwen32B'}


def load_data(model_name, dataset):
    """Select loading path and average_type based on data availability."""
    no_early_exit = dataset.startswith('bigbench') or model_name in _NO_EARLY_EXIT_MODELS

    # bigbench and some TSP combos only have ntokens_300
    use_ntokens_300 = (
        dataset.startswith('bigbench')
        or (dataset == 'TSP' and model_name in _NTOKENS_300_MODELS)
    )
    average_type = 'ntokens_300' if use_ntokens_300 else AVERAGE_TYPE

    if no_early_exit:
        return load_without_early_exit(model_name, dataset, average_type=average_type)
    return load_with_early_exit(model_name, dataset)

## AUC computation

In [10]:
records = []

for model_name in MODELS:
    for dataset in DATASETS:
        print(model_name, dataset)
        try:
            df, available_metrics = load_data(model_name, dataset)
        except FileNotFoundError as e:
            print(f'  [SKIP] {e}')
            continue

        for metric in available_metrics:
            metric_df = df.loc[df['metric_name'] == metric]
            if metric_df.empty:
                continue

            scores = pd.to_numeric(metric_df['value'], errors='coerce').values
            if metric in NEGATE:
                scores = -scores

            y = metric_df['accuracy'].astype(bool).values
            mask = np.isfinite(scores)
            y_valid, scores_valid = y[mask], scores[mask]

            auc = np.nan
            if mask.sum() == 0:
                print(f'  [WARN] no finite scores: {metric}')
            elif np.unique(y_valid).size < 2:
                print(f'  [WARN] single class: {metric}')
            else:
                try:
                    auc = roc_auc_score(y_valid, scores_valid)
                except Exception as e:
                    print(f'  [WARN] AUC error for {metric}: {e}')

            records.append({
                'model_name': model_name,
                'dataset':    dataset,
                'metric':     metric,
                'auc':        None if np.isnan(auc) else round(float(auc), 4),
            })

auc_df = pd.DataFrame(records)

phi4-reasoning-plus GPQA
phi4-reasoning-plus AIME2025
phi4-reasoning-plus TSP


/pfss/mlde/workspaces/mlde_wsp_PI_Roig/martinavilas/reasoning_traces/notebooks/../src/pre_utils.py:40: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(n=20, random_state=0))


phi4-reasoning-plus bigbench/fables
phi4-reasoning-plus bigbench/social-iqa
deepseek-r1-qwen14B GPQA
deepseek-r1-qwen14B AIME2025
deepseek-r1-qwen14B TSP


/pfss/mlde/workspaces/mlde_wsp_PI_Roig/martinavilas/reasoning_traces/notebooks/../src/pre_utils.py:40: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(n=20, random_state=0))


deepseek-r1-qwen14B bigbench/fables
deepseek-r1-qwen14B bigbench/social-iqa
qwen3-14B GPQA
qwen3-14B AIME2025
qwen3-14B TSP


/pfss/mlde/workspaces/mlde_wsp_PI_Roig/martinavilas/reasoning_traces/notebooks/../src/pre_utils.py:40: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(n=20, random_state=0))


qwen3-14B bigbench/fables
qwen3-14B bigbench/social-iqa
deepseek-r1-llama8B GPQA
deepseek-r1-llama8B AIME2025
deepseek-r1-llama8B TSP
deepseek-r1-llama8B bigbench/fables
deepseek-r1-llama8B bigbench/social-iqa
deepseek-r1-qwen32B GPQA
  [SKIP] [Errno 2] No such file or directory: '/pfss/mlde/workspaces/mlde_wsp_PI_Roig/martinavilas/reasoning_traces/results/internals/GPQA/deepseek-r1-qwen32B_ntokens_500_internals.pt'
deepseek-r1-qwen32B AIME2025
deepseek-r1-qwen32B TSP
deepseek-r1-qwen32B bigbench/fables
deepseek-r1-qwen32B bigbench/social-iqa
deepseek-r1-llama70B GPQA
deepseek-r1-llama70B AIME2025
deepseek-r1-llama70B TSP
deepseek-r1-llama70B bigbench/fables
  [SKIP] [Errno 2] No such file or directory: '/pfss/mlde/workspaces/mlde_wsp_PI_Roig/martinavilas/reasoning_traces/results/internals/bigbench/fables/deepseek-r1-llama70B_ntokens_300_internals.pt'
deepseek-r1-llama70B bigbench/social-iqa
  [SKIP] [Errno 2] No such file or directory: '/pfss/mlde/workspaces/mlde_wsp_PI_Roig/martinavi

In [11]:
output_path = PATH / 'results' / 'auc_all.csv'
output_path.parent.mkdir(parents=True, exist_ok=True)
auc_df.to_csv(output_path, index=False)
print(f'Saved to {output_path}')

Saved to /pfss/mlde/workspaces/mlde_wsp_PI_Roig/martinavilas/reasoning_traces/results/auc_all.csv


## Summary

In [12]:
summary = (
    auc_df.groupby('metric')['auc']
    .agg(['mean', 'std'])
    .rename(columns={'mean': 'AUC mean', 'std': 'AUC std'})
    .reindex(ALL_METRICS)
)
summary.index = summary.index.map(LABEL_MAP)
summary.round(3)

,AUC mean,AUC std
metric,,
Net Change,0.703,0.101
Cumulative Change,0.697,0.112
Aligned Change,0.701,0.099
Layer Mag,0.564,0.180
Layer Ang,0.584,0.186
Logit Margin,0.536,0.082
Entropy,0.448,0.131
Perplexity,0.467,0.078
